# Drieu, Todorova & Zugaro (2018) on CRCNS hc-18: theta sequences and sleep replay

Paper. C. Drieu, R. Todorova, M. Zugaro (2018), Science 362:675-679, doi:10.1126/science.aat2952; claims file `claims/drieu-2018.yaml` (version of record, and the author manuscript HAL hal-02330673v1 for the Methods and supplement).

Dataset. CRCNS hc-18 (doi:10.6080/K0Z899MM), `unversioned` (CRCNS does not version data sets). The only published copy is `hc-18/live/` in the bucket `crcnsarchive`, which is what the paper cites (data descriptor version 0.61); the MD5 sums of the files read are printed below. Streamed and reduced in memory by `drieu2018.py`, as in the first notebook.

Decoding follows the Methods (Bayesian trajectory decoding; Assessing trajectories) and the paper's scripts in the data set's `code.zip` (BatchSequenceReconstructNewProto.m, BatchSequenceQuadrants.m, BatchReplayReconstructNewProto.m, BatchCandidateEvents.m). The paper uses 5000 shuffles per event; this notebook uses 200 (see the deviation in the record): with about 30,000 theta cycles and 60,000 candidate events, 5000 shuffles would take about 10 hours here. 200 shuffles resolve Monte Carlo p-values to 0.005, well below the 0.05 used to call an event significant.

| claim | here |
|---|---|
| fig3b-average-theta-reconstruction | tested |
| figs5b-per-rat-reconstruction | tested |
| fig3c-theta-scores-above-chance | tested |
| fig3c-proportion-significant-theta | tested |
| fig3d-theta-slopes-steeper-active | tested |
| fig3e-joint-score-slope-theta | tested |
| fig3f-quadrant-scores | tested |
| figs9-control-decoding-quality | tested |
| fig4b-replay-scores-vs-baseline | tested |
| fig4c-replay-slopes-steeper-active | tested |
| fig4d-joint-score-slope-replay | tested |
| fig4e-replay-proportion-vs-baseline | tested |
| fig4f-forward-replay-after-active | tested |
| figs12-ripple-restricted | tested |
| fig3g-theta-self-consistency, fig4g-theta-replay-correlation-active | blocked, too_large (timed below) |
| figs7-control-cell-count, figs8-control-firing-rate, figs10-control-lap-variability | blocked, too_large (each re-decodes the theta cycles; see the record) |
| the travel, field, theta-LFP and sleep claims | in the first notebook |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import time, resource, gc
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.anova import AnovaRM
import drieu2018 as D

T_START = time.time()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
NSH = 200
BIN_M = D.TRACK_M / D.NBINS

In [ ]:
sums = D.checksums()
for n in [f"data/{r}.tar.gz" for r in D.RATS]:
    print(f"{sums[n]}  {n}")

In [ ]:
RAT, PREP, LFPS = {}, {}, {}
for name in D.RATS:
    t = time.time()
    r = D.load_rat(name)
    P = D.prepare(r)
    LF = D.reduce_lfp(r, P)
    r["lfp_shape"] = r["lfp"]["data"].shape
    del r["lfp"]
    RAT[name], PREP[name], LFPS[name] = r, P, LF
    gc.collect()
    print(f"{name}: {time.time() - t:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Decoding theta cycles

For each rat and travel condition, the decoder is trained on that condition's place cells (field reading a of the claims file) and applied to its theta cycles, six phase windows of pi/3 per cycle. Every analysed cycle gets a trajectory score and slope against 200 shuffles, and a quadrant score.

In [ ]:
def slope_ms(Dv, m, dur_s):
    """m/s from a line's displacement D (bins) over m windows lasting dur_s in total."""
    v_bins_per_win = Dv / (m - 1)
    return v_bins_per_win * BIN_M / (dur_s / m)

TH = {}
rows = []
t0 = time.time()
for k, name in enumerate(D.RATS):
    P, LF = PREP[name], LFPS[name]
    for j, c in enumerate(D.CONDS):
        rng = np.random.default_rng(1000 + 10 * k + j)
        th = D.decode_theta(P, LF, c)
        sc = D.score_events(th["mats"], NSH, rng)
        q = D.quadrant_scores(th["cent"])
        TH[(name, c)] = dict(th=th, sc=sc, q=q)
        mean_period = np.mean(np.diff(th["edges"][:, [0, 6]], axis=1)) if len(th["edges"]) else np.nan
        for e in range(len(th["mats"])):
            rows.append(dict(rat=name[6:9], cond=c, score=sc["score"][e], z=sc["z"][e], p=sc["p"][e], D=sc["D"][e],
                             dur=th["dur"][e], quad=q[e], t=th["edges"][e, 0],
                             slope_a=slope_ms(sc["D"][e], 6, th["dur"][e]), slope_b=slope_ms(sc["D"][e], 6, mean_period),
                             cm_per_cycle=sc["D"][e] / 5 * 6 * BIN_M * 100))
        print(f"{name[6:9]} {c}: place cells {th['n_cells']}, theta cycles {th['n_cycles_all']}, analysed {len(th['mats'])} "
              f"({time.time() - t0:.0f} s)", flush=True)
CY = pd.DataFrame(rows)

In [ ]:
print("Analysed theta cycles per rat and condition:")
print(CY.groupby(["rat", "cond"]).size().unstack().reindex(columns=D.CONDS).to_string())
print("Cycle duration (s): ", CY.dur.describe().round(3).to_dict())
print("Trajectory score range", CY.score.min().round(3), CY.score.max().round(3), "; Monte Carlo p range", CY.p.min(), CY.p.max())
print("Quadrant score range", np.nanmin(CY.quad).round(3), np.nanmax(CY.quad).round(3), "; NaN quadrant scores", int(CY.quad.isna().sum()))

## fig3b-average-theta-reconstruction

> Averaged over all theta cycles of all rats, the reconstruction of position across theta phase had trajectory score 0.17 and slope 6 cm/cycle in Passive 1, 0.23 and 48 cm/cycle in Active, and 0.19 and 21 cm/cycle in Passive 2.

Decoded probabilities centred on the animal's position, averaged over all analysed cycles of all rats, then scored. The slope is the best line's displacement over the six windows, in cm per cycle (displacement per window times 6).

In [ ]:
AVG = {}
rng = np.random.default_rng(2001)
for c in D.CONDS:
    allc = np.concatenate([TH[(n, c)]["th"]["cent"] for n in D.RATS])
    avg = np.nanmean(allc, 0)
    o = D.score_events([avg], 10, rng)
    AVG[c] = dict(score=o["score"][0], cm=o["D"][0] / 5 * 6 * BIN_M * 100, n=len(allc))
exp = {"P1": (0.17, 6), "A": (0.23, 48), "P2": (0.19, 21)}
tolr = {"P1": (0.017, 3), "A": (0.023, 4.8), "P2": (0.019, 3)}
for c in D.CONDS:
    a = AVG[c]
    print(f"{c}: cycles {a['n']}, score {a['score']:.3f} (expected {exp[c][0]} +/- {tolr[c][0]}), slope {a['cm']:.1f} cm/cycle "
          f"(expected {exp[c][1]} +/- {tolr[c][1]})")

**Result: partially reproduced.** The scores 0.178, 0.245 and 0.205 are within tolerance, and so is the Active slope (46.1 cm/cycle). The passive slopes, 23.0 and 17.3 cm/cycle, are outside it (expected 6 and 21). Active has the highest score and slope.

## figs5b-per-rat-reconstruction

> Across the five rats, the trajectory scores, slopes and quadrant scores of the average theta reconstructions differed between conditions (repeated-measures ANOVA, P = 0.008, 0.007 and 0.004), higher in Active.

In [ ]:
rows = []
rng = np.random.default_rng(2002)
for name in D.RATS:
    for c in D.CONDS:
        cent = TH[(name, c)]["th"]["cent"]
        avg = np.nanmean(cent, 0)
        o = D.score_events([avg], 10, rng)
        q = D.quadrant_scores(avg[None])[0]
        rows.append(dict(rat=name[6:9], cond=c, score=o["score"][0], slope=o["D"][0] / 5 * 6 * BIN_M * 100, quad=q))
PR = pd.DataFrame(rows)
print(PR.round(3).pivot(index="rat", columns="cond").to_string())
for col in ["score", "slope", "quad"]:
    res = AnovaRM(PR, col, "rat", within=["cond"]).fit().anova_table
    print(f"{col}: means {PR.groupby('cond')[col].mean().reindex(D.CONDS).round(3).to_dict()}; RM-ANOVA F = {res['F Value'].iloc[0]:.2f}, P = {res['Pr > F'].iloc[0]:.4f}")

**Result: partially reproduced.** Over rats, the score (P = 0.0075) and the quadrant score (P = 0.028) differ, with Active highest. The slope does not reach 0.05 (P = 0.079), though its Active mean is highest (47.8 against 19.0 and 19.6 cm/cycle).

## fig3c-theta-scores-above-chance

> Normalized trajectory scores of theta sequences were better than chance in all three conditions, and differed between conditions (Kruskal-Wallis, P < 10^-23).

In [ ]:
g = {c: CY[CY.cond == c].z.values for c in D.CONDS}
for c in D.CONDS:
    w = stats.wilcoxon(g[c])
    print(f"{c}: n {len(g[c])}, median normalized score {np.median(g[c]):.3f}, signed-rank vs 0 P = {w.pvalue:.1e}")
h, p = stats.kruskal(*g.values())
print(f"KW H = {h:.1f}, P = {p:.1e}")
for a, b in [("P1", "A"), ("A", "P2"), ("P1", "P2")]:
    pp = stats.mannwhitneyu(g[a], g[b]).pvalue
    print(f"   rank-sum {a} vs {b}: P (Bonferroni x3) = {min(1, 3 * pp):.1e}; medians {np.median(g[a]):.3f} vs {np.median(g[b]):.3f}")
print("per rat median normalized score:")
print(CY.groupby(["rat", "cond"]).z.median().unstack().reindex(columns=D.CONDS).round(3).to_string())

**Result: reproduced.** Median normalized scores (0.49, 0.64 and 0.65) are above 0 in every condition (signed-rank P < 10^-200), and Kruskal-Wallis gives P = 4.2 x 10^-25. The difference comes from Passive 1 being lower; Active and Passive 2 do not differ (P = 1).

## fig3c-proportion-significant-theta

> The proportion of significant theta sequences (significant score, positive slope) was 7.95% in Passive 1, 13.61% in Active and 9.05% in Passive 2, each above shuffled control, and Active exceeded both passive conditions.

Shuffled control: for each shuffle j, every event's j-th shuffled score is tested against its other shuffles with the same rule (p <= 0.05, and positive slope); the 200 shuffled proportions give the Monte Carlo test.

In [ ]:
def shuffled_props(keys, slope_rule):
    out = []
    for j in range(NSH):
        k_sig, n = 0, 0
        for key in keys:
            sc = TH[key]["sc"] if key in TH else REP[key]["sc"]
            if len(sc["score"]) == 0:
                continue
            sh = sc["sh_score"]
            pj = (np.sum(sh > sh[:, j:j + 1], 1)) / (NSH - 1)
            ok = pj <= 0.05
            if slope_rule == "pos":
                ok &= sc["sh_D"][:, j] > 0
            k_sig += ok.sum()
            n += len(sh)
        out.append(k_sig / n)
    return np.array(out)

def prop_test(k1, n1, k2, n2):
    p = (k1 + k2) / (n1 + n2)
    z = (k1 / n1 - k2 / n2) / np.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    return z, 2 * stats.norm.sf(abs(z))

In [ ]:
SIGTH = {}
for rule, lab in [("pos", "significant and positive slope"), ("any", "significant, any slope")]:
    print(lab)
    for c in D.CONDS:
        sub = CY[CY.cond == c]
        sig = (sub.p <= 0.05) & ((sub.D > 0) if rule == "pos" else True)
        shp = shuffled_props([(n, c) for n in D.RATS], rule)
        SIGTH[(c, rule)] = (int(sig.sum()), len(sub))
        print(f"   {c}: {sig.sum()}/{len(sub)} = {100 * sig.mean():.2f}%; shuffled control mean {100 * shp.mean():.2f}% (max {100 * shp.max():.2f}%), "
              f"Monte Carlo P = {np.mean(shp >= sig.mean()):.3f}")
    for a, b in [("A", "P1"), ("A", "P2"), ("P1", "P2")]:
        z, p = prop_test(*SIGTH[(a, rule)], *SIGTH[(b, rule)])
        print(f"   binomial proportion test {a} vs {b}: z = {z:.2f}, P = {p:.1e}")
print("expected: 7.95, 13.61, 9.05 (positive slope); 14.66, 20.20, 16.37 (any slope); tolerance 10%")
print("per rat, % significant with positive slope:")
print((CY.assign(s=(CY.p <= 0.05) & (CY.D > 0)).groupby(["rat", "cond"]).s.mean().unstack().reindex(columns=D.CONDS) * 100).round(2).to_string())

**Result: not reproduced.** With the slope rule the proportions are 6.93%, 11.39% and 9.90%, of which only Passive 2 is within tolerance. Without it they are 14.30%, 17.88% and 18.07%, of which only Passive 1 is. All exceed the shuffled control. Active exceeds Passive 1 (P = 1.4 x 10^-21), but Passive 2 only at P = 0.0048, so the criterion's P < 0.001 fails.

## fig3d-theta-slopes-steeper-active

> Slopes of significant theta sequences were steeper during active travel than in either passive condition, while the two passive conditions did not differ.

In [ ]:
sig = CY[CY.p <= 0.05]
for col in ["slope_a", "slope_b"]:
    for kind in ["signed", "absolute"]:
        g = {c: (sig[sig.cond == c][col].values if kind == "signed" else np.abs(sig[sig.cond == c][col].values)) for c in D.CONDS}
        h, p = stats.kruskal(*g.values())
        print(f"{col} ({kind}): n {[len(v) for v in g.values()]}, median (m/s) {[round(np.median(v), 2) for v in g.values()]}; KW P = {p:.1e}")
        for a, b in [("P1", "A"), ("A", "P2"), ("P1", "P2")]:
            ks = stats.ks_2samp(g[a], g[b])
            print(f"   KS {a} vs {b}: D = {ks.statistic:.3f}, P = {ks.pvalue:.1e}")
print("slope_a range (m/s):", sig.slope_a.min().round(1), sig.slope_a.max().round(1))

**Result: partially reproduced.** Signed slopes of significant sequences are steeper in Active than in Passive 1 and Passive 2 (KS P = 1.4 x 10^-17 and 3.6 x 10^-12), but the passive pair also differs (P = 0.019). Absolute slopes leave the passive pair alike (P = 0.62), but the Active comparisons reach only P = 5.6 x 10^-6 and 2.6 x 10^-4.

## fig3e-joint-score-slope-theta

> Joint analysis of trajectory scores and slopes showed a much greater proportion of high-score, high-slope theta sequences during active travel.

Z-proportions (Methods, Z-proportions): the score-slope matrix of event proportions, each cell z-scored against the same cell of the shuffled matrices (here 200, one per shuffle index). Grid 0.05 score by 3 m/s slope, and half and double. High-value region (criterion): score above the 95th percentile of the pooled shuffled scores and slope above 3 m/s. Slopes with reading (a).

In [ ]:
def zprop(scores, slopes, sh_scores, sh_slopes, ds, dv, region):
    se = np.arange(0, 1.0001 + ds, ds)
    ve = np.arange(-60, 60 + dv, dv)
    H = np.histogram2d(scores, np.clip(slopes, -59.9, 59.9), [se, ve])[0] / len(scores)
    Hs = np.array([np.histogram2d(sh_scores[:, j], np.clip(sh_slopes[:, j], -59.9, 59.9), [se, ve])[0] / len(scores) for j in range(sh_scores.shape[1])])
    with np.errstate(invalid="ignore", divide="ignore"):
        Z = (H - Hs.mean(0)) / Hs.std(0)
    sc_c = (se[:-1] + se[1:]) / 2
    v_c = (ve[:-1] + ve[1:]) / 2
    reg = region(sc_c[:, None], v_c[None, :])
    Zr = np.where(reg & np.isfinite(Z) & (Z > 0), Z, 0).sum()
    return Zr

def theta_shuffled_slopes(key):
    th, sc = TH[key]["th"], TH[key]["sc"]
    return slope_ms(sc["sh_D"], 6, th["dur"][:, None])

In [ ]:
shs_all = np.concatenate([TH[k]["sc"]["sh_score"].ravel() for k in TH])
thr95 = np.percentile(shs_all, 95)
print(f"95th percentile of pooled shuffled theta scores: {thr95:.3f}")
region = lambda s, v: (s > thr95) & (v > 3)
for ds, dv in [(0.05, 3), (0.025, 1.5), (0.1, 6)]:
    zr = {}
    for c in D.CONDS:
        keys = [(n, c) for n in D.RATS]
        sc = np.concatenate([TH[k]["sc"]["score"] for k in keys])
        sl = np.concatenate([CY[(CY.rat == k[0][6:9]) & (CY.cond == c)].slope_a.values for k in keys])
        shs = np.concatenate([TH[k]["sc"]["sh_score"] for k in keys])
        shv = np.concatenate([theta_shuffled_slopes(k) for k in keys])
        zr[c] = zprop(sc, sl, shs, shv, ds, dv, region)
    print(f"grid {ds} x {dv} m/s: summed positive z-proportion in the high-value region {dict((c, round(v, 1)) for c, v in zr.items())}")
cnt = {}
for c in D.CONDS:
    sub = CY[CY.cond == c]
    k = int(((sub.score > thr95) & (sub.slope_a > 3)).sum())
    cnt[c] = (k, len(sub))
    print(f"{c}: events in region {k}/{len(sub)} = {100 * k / len(sub):.2f}%")
for a in ["P1", "P2"]:
    z, p = prop_test(*cnt["A"], *cnt[a])
    print(f"   binomial proportion test A vs {a}: z = {z:.2f}, P = {p:.1e}")

**Result: reproduced, resting on the claims file's region.** The summed positive z-proportion of the high-value region is largest in Active on all three grids (303.8 against 39.7 and 123.4 on the main grid). The proportion of events in the region is 8.82% against 0.25% and 2.99% (P < 10^-45).

## fig3f-quadrant-scores

> Quadrant scores of theta cycles were greater during active travel, remained very low in passive travel, and were higher in Passive 2 than in Passive 1 (Kruskal-Wallis, P < 10^-24).

In [ ]:
g = {c: CY[CY.cond == c].quad.dropna().values for c in D.CONDS}
h, p = stats.kruskal(*g.values())
print(f"n {[len(v) for v in g.values()]}; median quadrant score {[round(np.median(v), 4) for v in g.values()]}; mean {[round(np.mean(v), 4) for v in g.values()]}; KW P = {p:.1e}")
for a, b in [("A", "P1"), ("A", "P2"), ("P2", "P1")]:
    pp = stats.mannwhitneyu(g[a], g[b]).pvalue
    print(f"   rank-sum {a} vs {b}: P (Bonferroni x3) = {min(1, 3 * pp):.1e}; median {np.median(g[a]):.4f} vs {np.median(g[b]):.4f}")
print(CY.groupby(["rat", "cond"]).quad.median().unstack().reindex(columns=D.CONDS).round(4).to_string())

**Result: reproduced.** Kruskal-Wallis gives P = 2.7 x 10^-47, Active is above both passive conditions (P = 5.7 x 10^-46 and 5.3 x 10^-23), and Passive 2 is above Passive 1 (P = 0.0013).

## figs9-control-decoding-quality

> With passive-travel theta cycles restricted to low decoding-error periods matching Active, slopes and quadrant scores still differed between conditions, while normalized scores no longer did (Kruskal-Wallis, P > 0.05).

Per rat and condition, positions decoded in 500-ms bins of moving travel with the condition's decoder; error = circular distance (m) between the decoded maximum and the actual position. In Passive 1 and Passive 2, the highest-error bin is dropped repeatedly until the median error is below the Active median (criterion's reading). Theta cycles starting in kept bins are kept.

In [ ]:
ERR = {}
keep_bins = {}
for name in D.RATS:
    P = PREP[name]
    ok = np.isfinite(P["lin_unw"])
    tt, lu = P["t"][ok], P["lin_unw"][ok]
    meds = {}
    for c in D.CONDS:
        cells = D.place_cells(P, c, "a")
        px, lam = D.train_decoder(P, c, cells)
        edges = []
        for a, b in P["mov"]:
            lo, hi = max(a, P["ep"]["travel"][c][0]), min(b, P["ep"]["travel"][c][1])
            if hi - lo >= 0.5:
                edges.append(np.arange(lo, hi - 0.5 + 1e-9, 0.5))
        starts = np.concatenate(edges) if edges else np.zeros(0)
        cnt = np.array([[np.sum((st >= s) & (st < s + 0.5)) for s in starts] for st in [P["cond"][c]["cells"][i]["spk_t"] for i in cells]])
        post = D.decode(cnt, np.full(len(starts), 0.5), px, lam)
        dec = (np.nanargmax(np.nan_to_num(post, nan=-1), 1) + 0.5) / D.NBINS
        act = np.mod(np.interp(starts + 0.25, tt, lu), 1)
        err = np.abs((dec - act + 0.5) % 1 - 0.5) * D.TRACK_M
        err[np.isnan(post[:, 0])] = np.nan
        ERR[(name, c)] = (starts, err)
        meds[c] = np.nanmedian(err)
    for c in D.CONDS:
        starts, err = ERR[(name, c)]
        kept = np.isfinite(err)
        if c != "A":
            order = np.argsort(-np.nan_to_num(err, nan=-1))
            k = 0
            while np.median(err[kept]) >= meds["A"] and kept.sum() > 1:
                kept[order[k]] = False
                k += 1
        keep_bins[(name, c)] = (starts[kept], err[kept])
    print(name[6:9], "median error (m) before:", {c: round(meds[c], 3) for c in D.CONDS},
          "after:", {c: round(np.median(keep_bins[(name, c)][1]), 3) for c in D.CONDS},
          "bins kept:", {c: f"{len(keep_bins[(name, c)][0])}/{np.isfinite(ERR[(name, c)][1]).sum()}" for c in D.CONDS})
rows = []
for name in D.RATS:
    for c in D.CONDS:
        starts = keep_bins[(name, c)][0]
        sub = CY[(CY.rat == name[6:9]) & (CY.cond == c)]
        j = np.searchsorted(starts, sub.t.values, side="right") - 1
        inbin = (j >= 0) & (sub.t.values < starts[np.clip(j, 0, None)] + 0.5) if len(starts) else np.zeros(len(sub), bool)
        rows.append(sub[inbin])
CY9 = pd.concat(rows)
errs = {c: np.concatenate([keep_bins[(n, c)][1] for n in D.RATS]) for c in D.CONDS}
print("pooled error after restriction, mean +/- s.e.m. (m):", {c: f"{errs[c].mean():.3f} +/- {stats.sem(errs[c]):.3f}" for c in D.CONDS},
      " expected 0.802, 0.80, 0.802 (+/-0.018, 0.023, 0.014)")
print("pooled median error after restriction (m):", {c: round(np.median(errs[c]), 3) for c in D.CONDS})
print("cycles kept:", CY9.groupby("cond").size().reindex(D.CONDS).to_dict(), "of", CY.groupby("cond").size().reindex(D.CONDS).to_dict())
s9 = CY9[CY9.p <= 0.05]
g = {c: s9[s9.cond == c].slope_a.values for c in D.CONDS}
print(f"significant slopes (m/s, reading a): n {[len(v) for v in g.values()]}, medians {[round(np.median(v), 2) for v in g.values()]}; KW P = {stats.kruskal(*g.values()).pvalue:.1e}")
for a, b in [("P1", "A"), ("A", "P2"), ("P1", "P2")]:
    print(f"   KS {a} vs {b}: P = {stats.ks_2samp(g[a], g[b]).pvalue:.1e}")
gz = {c: CY9[CY9.cond == c].z.values for c in D.CONDS}
print(f"normalized score medians {[round(np.median(v), 3) for v in gz.values()]}; KW P = {stats.kruskal(*gz.values()).pvalue:.1e}")
gq = {c: CY9[CY9.cond == c].quad.dropna().values for c in D.CONDS}
print(f"quadrant score medians {[round(np.median(v), 4) for v in gq.values()]}; KW P = {stats.kruskal(*gq.values()).pvalue:.1e}")
for a in ["P1", "P2"]:
    print(f"   quadrant rank-sum A vs {a}: P (Bonferroni x3) = {min(1, 3 * stats.mannwhitneyu(gq['A'], gq[a]).pvalue):.1e}")

**Result: partially reproduced.** After matching median errors, slopes (KS P = 7.5 x 10^-11 and 6.3 x 10^-10) and quadrant scores (P = 2.0 x 10^-37) still differ, with Active highest. The normalized scores also still differ (P = 9.1 x 10^-15), where the paper finds no difference. The mean errors, 0.41 to 0.54 m, are outside the expected 0.80 m.

## Replay: candidate events in sleep

Candidate events in SWS of each post-travel sleep, from the place cells of the preceding travel (field reading a; threshold mean and s.d. over SWS, as BatchCandidateEvents.m), and in baseline sleep from the same place cells; both decoded with that travel's decoder in 20-ms windows and scored against 200 shuffles.

In [ ]:
SLEEP_OF = {"P1": "S1", "A": "SA", "P2": "S2"}
REP = {}
rows = []
t0 = time.time()
for k, name in enumerate(D.RATS):
    P, LF = PREP[name], LFPS[name]
    for j, c in enumerate(D.CONDS):
        cells = D.place_cells(P, c, "a")
        px, lam = D.train_decoder(P, c, cells)
        spikes = [RAT[name]["units"][i]["times"] for i in cells]
        allspk = np.sort(np.concatenate(spikes))
        for kind, s in [("post", SLEEP_OF[c]), ("base", "SB")]:
            sl = LF["sleep"][s]
            ev = D.candidate_events(allspk, sl["interval"], sl["sws"], norm="sws")
            mats, keep = D.replay_matrices(ev, spikes, px, lam)
            rng = np.random.default_rng(5000 + 100 * k + 10 * j + (kind == "base"))
            sc = D.score_events(mats, NSH, rng)
            REP[(name, c, kind)] = dict(sc=sc, ev=ev[keep], n_candidates=len(ev))
            rip = sl["ripples"]
            for e in range(len(mats)):
                a, pk, b = ev[keep[e]]
                m = len(mats[e])
                ov = np.any((rip[:, 0] <= b) & (rip[:, 2] >= a)) if len(rip) else False
                pin = np.any((rip[:, 0] <= pk) & (rip[:, 2] >= pk)) if len(rip) else False
                rows.append(dict(rat=name[6:9], cond=c, kind=kind, score=sc["score"][e], z=sc["z"][e], p=sc["p"][e], D=sc["D"][e],
                                 m=m, slope=sc["D"][e] / (m - 1) * BIN_M / 0.02, rip_overlap=ov, rip_peak=pin))
        print(f"{name[6:9]} {c}: place cells {len(cells)}; post events {REP[(name, c, 'post')]['n_candidates']} -> {len(REP[(name, c, 'post')]['sc']['score'])} decoded; "
              f"baseline {REP[(name, c, 'base')]['n_candidates']} -> {len(REP[(name, c, 'base')]['sc']['score'])} ({time.time() - t0:.0f} s)", flush=True)
EV = pd.DataFrame(rows)
# score relative to baseline, per rat and decoder (Methods, Assessing trajectories, paragraph 3)
EV["rel"] = np.nan
for (rat, c), sub in EV.groupby(["rat", "cond"]):
    b = sub[sub.kind == "base"].score
    q1, med, q3 = np.percentile(b, [25, 50, 75])
    EV.loc[sub.index, "rel"] = (sub.score - med) / (q3 - q1)

In [ ]:
print("Decoded candidate events per rat, decoder and sleep (post / baseline):")
print(EV.groupby(["rat", "cond", "kind"]).size().unstack([1, 2]).to_string())
print("windows per event:", EV.m.describe().round(1).to_dict())
print("fraction of events overlapping a ripple:", EV.groupby("kind").rip_overlap.mean().round(3).to_dict())

## fig4b-replay-scores-vs-baseline

> Replay trajectory scores, relative to baseline sleep, were improved only in sleep after Active and after Passive 2 (Kruskal-Wallis, P < 10^-17).

In [ ]:
def fig4b(E, label):
    post = E[E.kind == "post"]
    g = {c: post[post.cond == c].rel.values for c in D.CONDS}
    print(label)
    for c in D.CONDS:
        w = stats.wilcoxon(g[c])
        print(f"   sleep after {c}: n {len(g[c])}, median score relative to baseline {np.median(g[c]):.3f}, signed-rank vs 0 P = {w.pvalue:.1e}")
    print(f"   KW P = {stats.kruskal(*g.values()).pvalue:.1e}")
    for a, b in [("P1", "A"), ("A", "P2"), ("P1", "P2")]:
        print(f"   rank-sum {a} vs {b}: P (Bonferroni x3) = {min(1, 3 * stats.mannwhitneyu(g[a], g[b]).pvalue):.1e}")
    print("   per rat median:", post.groupby(["rat", "cond"]).rel.median().unstack().reindex(columns=D.CONDS).round(3).to_dict("index"))

In [ ]:
fig4b(EV, "all decoded candidate events")

**Result: not reproduced.** Scores relative to baseline are improved after Active (median 0.274, P = 6 x 10^-223) and slightly after Passive 1 (0.028, P = 8 x 10^-20), but not after Passive 2 (-0.055, below 0, P = 0.019).

## fig4c-replay-slopes-steeper-active

> Replay slopes were steeper in sleep after Active than after either passive travel, while sleep after Passive 1 and Passive 2 did not differ.

In [ ]:
def fig4c(E, label):
    post = E[E.kind == "post"]
    for sel, lab in [(post, "all events"), (post[post.p <= 0.05], "significant events")]:
        g = {c: np.abs(sel[sel.cond == c].slope.values) for c in D.CONDS}
        gs = {c: sel[sel.cond == c].slope.values for c in D.CONDS}
        print(f"{label}, {lab}: n {[len(v) for v in g.values()]}, median |slope| (m/s) {[round(np.median(v), 2) for v in g.values()]}; "
              f"KW P = {stats.kruskal(*g.values()).pvalue:.1e}")
        for a, b in [("P1", "A"), ("A", "P2"), ("P1", "P2")]:
            print(f"   KS signed slopes {a} vs {b}: P = {stats.ks_2samp(gs[a], gs[b]).pvalue:.1e}; KS |slopes| P = {stats.ks_2samp(g[a], g[b]).pvalue:.1e}")

In [ ]:
fig4c(EV, "all decoded")

**Result: partially reproduced, resting on an assumption.** On significant events, absolute slopes are steepest after Active (5.7 against 2.8 and 5.1 m/s, Kruskal-Wallis P = 3.9 x 10^-5), with KS P = 3.1 x 10^-5 and 8.1 x 10^-4 for the Active comparisons. The passive pair differs (P = 0.030). On all events there is no effect (P = 0.14).

## fig4d-joint-score-slope-replay

> Joint analysis of replay trajectory scores and slopes showed a much greater proportion of high-score, high-slope events in sleep after Active.

As fig3e, on post-travel events, with |slope| above 3 m/s. The proportion ratio of fig. S13D is shown as the ratio of the post-sleep proportion in the region to the baseline one.

In [ ]:
shr = np.concatenate([REP[k]["sc"]["sh_score"].ravel() for k in REP if k[2] == "post" and len(REP[k]["sc"]["score"])])
thr95r = np.percentile(shr, 95)
print(f"95th percentile of pooled shuffled replay scores: {thr95r:.3f}")
regr = lambda s, v: (s > thr95r) & (np.abs(v) > 3)
for ds, dv in [(0.05, 3), (0.025, 1.5), (0.1, 6)]:
    zr = {}
    for c in D.CONDS:
        keys = [(n, c, "post") for n in D.RATS if len(REP[(n, c, "post")]["sc"]["score"])]
        sc = np.concatenate([REP[k]["sc"]["score"] for k in keys])
        ms = np.concatenate([REP[k]["sc"]["m"] for k in keys])
        sl = np.concatenate([REP[k]["sc"]["D"] for k in keys]) / (ms - 1) * BIN_M / 0.02
        shs = np.concatenate([REP[k]["sc"]["sh_score"] for k in keys])
        shv = np.concatenate([REP[k]["sc"]["sh_D"] for k in keys]) / (ms[:, None] - 1) * BIN_M / 0.02
        zr[c] = zprop(sc, sl, shs, shv, ds, dv, regr)
    print(f"grid {ds} x {dv} m/s: summed positive z-proportion in the high-value region, sleep after {dict((c, round(v, 1)) for c, v in zr.items())}")
cnt = {}
for c in D.CONDS:
    for kind in ["post", "base"]:
        sub = EV[(EV.cond == c) & (EV.kind == kind)]
        cnt[(c, kind)] = (int(((sub.score > thr95r) & (sub.slope.abs() > 3)).sum()), len(sub))
    k, n = cnt[(c, "post")]
    kb, nb = cnt[(c, "base")]
    print(f"after {c}: in region {k}/{n} = {100 * k / n:.2f}%; baseline {kb}/{nb} = {100 * kb / nb:.2f}%; proportion ratio {(k / n) / (kb / nb):.2f}")
for a in ["P1", "P2"]:
    z, p = prop_test(*cnt[("A", "post")], *cnt[(a, "post")])
    print(f"   binomial proportion test after A vs after {a}: z = {z:.2f}, P = {p:.1e}")

**Result: reproduced, resting on the claims file's region.** The summed positive z-proportion is largest after Active on all grids (328.6 against 76.1 and 127.8). 11.08% of events after Active fall in the region, against 0.61% and 3.25% (P < 10^-55).

## fig4e-replay-proportion-vs-baseline

> Relative to baseline sleep, the proportion of significant replay events was not increased after Passive 1, was boosted after Active, and was intermediate but significantly increased after Passive 2.

In [ ]:
def fig4e(E, label):
    print(label)
    for rule in ["a", "b"]:
        for c in D.CONDS:
            po = E[(E.cond == c) & (E.kind == "post")]
            ba = E[(E.cond == c) & (E.kind == "base")]
            sp = (po.p <= 0.05) & ((po.D > 0) if rule == "b" else True)
            sb = (ba.p <= 0.05) & ((ba.D > 0) if rule == "b" else True)
            z, p = prop_test(sp.sum(), len(po), sb.sum(), len(ba))
            print(f"   reading {rule}, after {c}: {sp.sum()}/{len(po)} = {100 * sp.mean():.2f}% vs baseline {sb.sum()}/{len(ba)} = {100 * sb.mean():.2f}%; "
                  f"ratio {sp.mean() / sb.mean():.2f}; binomial z = {z:.2f}, P = {p:.1e}")

In [ ]:
fig4e(EV, "all decoded candidate events")
print("per rat, % significant (reading a), post / baseline:")
print((EV.assign(s=EV.p <= 0.05).groupby(["rat", "cond", "kind"]).s.mean().unstack([1, 2]) * 100).round(1).to_string())

**Result: not reproduced.** Relative to baseline, significant events are not increased after Passive 1 (ratio 1.05, P = 0.27) and are increased after Active (1.26, P = 4.4 x 10^-7). After Passive 2 they are decreased (0.89, P = 0.021), not increased.

## fig4f-forward-replay-after-active

> Forward and reverse replay events were equally common in sleep after passive travel, while sleep after Active held more forward than reverse events.

In [ ]:
def fig4f(E, label):
    print(label)
    for c in D.CONDS:
        po = E[(E.cond == c) & (E.kind == "post") & (E.p <= 0.05)]
        ba = E[(E.cond == c) & (E.kind == "base") & (E.p <= 0.05)]
        fw, n = int((po.D > 0).sum()), len(po)
        rv = int((po.D < 0).sum())
        z = (fw / n - 0.5) / np.sqrt(0.25 / n)
        pa = 2 * stats.norm.sf(abs(z))
        fb, nb = int((ba.D > 0).sum()), len(ba)
        zb, pb = prop_test(fw, n, fb, nb)
        print(f"   after {c}: forward {fw}, reverse {rv}, zero {n - fw - rv}, of {n} significant; forward proportion {fw / n:.3f}; "
              f"(a) vs 0.5: z = {z:.2f}, P = {pa:.1e}; (b) vs baseline forward proportion {fb}/{nb} = {fb / max(nb, 1):.3f}: z = {zb:.2f}, P = {pb:.1e}")

In [ ]:
fig4f(EV, "all decoded candidate events")

**Result: partially reproduced.** After Active, forward events outnumber reverse ones (514 against 397, P = 0.0051), weaker than the criterion's P < 0.001. After the passive travels the proportions do not differ from 0.5 (P = 0.44 and 0.054).

## figs12-ripple-restricted

> Restricting candidate events to ripples left the replay results unchanged; scores relative to baseline, slopes, the boost after Active and the forward bias after Active all held.

Readings: (a) events that overlap a detected ripple; (b) events whose peak falls inside a ripple. Scores relative to baseline recomputed on the restricted baseline events.

In [ ]:
for col, lab in [("rip_overlap", "reading (a), any overlap with a ripple"), ("rip_peak", "reading (b), event peak inside a ripple")]:
    E = EV[EV[col]].copy()
    for (rat, c), sub in E.groupby(["rat", "cond"]):
        b = sub[sub.kind == "base"].score
        q1, med, q3 = np.percentile(b, [25, 50, 75])
        E.loc[sub.index, "rel"] = (sub.score - med) / (q3 - q1)
    print("=" * 20, lab, "; events kept", len(E), "of", len(EV))
    fig4b(E, lab)
    fig4c(E, lab)
    fig4e(E, lab)
    fig4f(E, lab)

**Result: not reproduced.** On ripple-restricted events none of the four criteria holds. After Passive 1 scores are above baseline (P = 7 x 10^-13), and after Passive 2 neither scores nor proportions are increased. The slope tests are far above the legend's bounds (P = 0.0044 and 0.007), and the forward bias after Active is not significant (P = 0.16). The after-Active score and proportion effects hold.

## Pairwise bias correlations (fig3g, fig4g): cost

Not run. The criterion compares every pair of sequences within a condition (and every theta-replay pair), each against a surrogate of shuffled spike order. The number of pairs and the time of one pair with 1000 surrogates are printed; the claims are blocked as too large.

In [ ]:
def bias_matrix(order_cells, n):
    B = np.zeros((n, n))
    for a in range(len(order_cells)):
        for b in range(a + 1, len(order_cells)):
            B[order_cells[a], order_cells[b]] += 1
    return (B - B.T)

rng = np.random.default_rng(7)
n = 30
seqs = [rng.permutation(n)[:8] for _ in range(2)]
t = time.time()
for _ in range(1000):
    b1 = bias_matrix(seqs[0], n); b2 = bias_matrix(rng.permutation(seqs[1]), n)
    common = np.intersect1d(seqs[0], seqs[1])
    x, y = b1[np.ix_(common, common)].ravel(), b2[np.ix_(common, common)].ravel()
    _ = x @ y / (np.linalg.norm(x) * np.linalg.norm(y) + 1e-12)
per_pair = time.time() - t
npairs_theta = sum(len(v) * (len(v) - 1) / 2 for v in [CY[(CY.rat == r) & (CY.cond == c)] for r in CY.rat.unique() for c in D.CONDS])
npairs_rep = sum(len(CY[(CY.rat == r) & (CY.cond == c)]) * len(EV[(EV.rat == r) & (EV.cond == c)]) for r in CY.rat.unique() for c in D.CONDS)
print(f"one pair with 1000 surrogates: {per_pair:.3f} s; theta-theta pairs within condition and rat: {npairs_theta:.3g}; theta-replay pairs: {npairs_rep:.3g}")
print(f"estimated time: {per_pair * (npairs_theta + npairs_rep) / 3600 / 4:.0f} hours on 4 cores")

## Summary

| claim | outcome |
|---|---|
| fig3b-average-theta-reconstruction | partially reproduced |
| figs5b-per-rat-reconstruction | partially reproduced |
| fig3c-theta-scores-above-chance | reproduced |
| fig3c-proportion-significant-theta | not reproduced |
| fig3d-theta-slopes-steeper-active | partially reproduced |
| fig3e-joint-score-slope-theta | reproduced (claims file's region) |
| fig3f-quadrant-scores | reproduced |
| figs9-control-decoding-quality | partially reproduced |
| fig4b-replay-scores-vs-baseline | not reproduced |
| fig4c-replay-slopes-steeper-active | partially reproduced (assumed significant events) |
| fig4d-joint-score-slope-replay | reproduced (claims file's region) |
| fig4e-replay-proportion-vs-baseline | not reproduced |
| fig4f-forward-replay-after-active | partially reproduced |
| figs12-ripple-restricted | not reproduced |
| fig3g, fig4g | blocked, too large |
| figs7, figs8, figs10 | blocked, too large for this run |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Differences from the paper's method. 200 shuffles per event instead of 5000. Place cells are those of the Methods' definition without a minimum field size, more numerous than the paper's. 250 position bins as the claims file gives (the scripts default to 200). The linearization, LFP channel and theta cycles are rebuilt, since the paper's files are not in the data set.

About the dataset. The theta sequences and the selective after-Active replay effects are present in these data: quadrant scores, z-proportions and replay scores relative to baseline all favour Active. The Passive 2 replay increase and the Active forward bias at the paper's level are not found. Rat 272 shows no Active theta-sequence effect.


In [ ]:
print(f"notebook runtime {(time.time() - T_START) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")